Configure the project and run folders in the code before running. Supply your own image data and saved checkpoints.

# JMuBEN external evaluation

Before running, confirm from the JMuBEN annotation documentation that the two extracted folders contain healthy leaves and coffee leaf rust respectively. Use the same screened JMuBEN images for all nine checkpoints. No model selection or threshold adjustment uses JMuBEN.

Cell 1 screens the JMuBEN photos against RoCoLe by exact file hash and a 63-bit perceptual hash. The screen excludes exact matches and candidate near duplicates within Hamming distance 6. It does not certify leaf identity or catch every near duplicate. Review the exclusion counts before running Cell 2.

Cell 2 evaluates the nine saved checkpoints at threshold 0.5 and writes per-image probabilities, test metrics, and paired comparisons to Drive.


In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from scipy.fft import dctn
from sklearn.metrics import f1_score
from google.colab import drive

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')
project = drive_root / 'YOUR_ROCOLE_PROJECT'
split = pd.read_csv(project / 'results/splits/combined_group_split.csv')
external_root = drive_root / 'YOUR_JMUBEN_PROJECT/dataset/extracted'
class_folders = {'healthy': external_root / 'healthy/Healthy',
                 'rust': external_root / 'rust/Leaf rust'}
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
sources = []
for label, folder in class_folders.items():
    assert folder.is_dir(), f'Missing JMuBEN folder: {folder}'
    sources.extend((path, label) for path in sorted(folder.rglob('*'))
                   if path.is_file() and path.suffix.lower() in extensions)
assert len(split) == 1389 and split.image_path.is_unique
assert {label for _, label in sources} == set(class_folders)
assert tf.config.list_physical_devices('GPU'), 'Select a GPU runtime.'

def fingerprint(path):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    with Image.open(path) as image:
        pixels = np.asarray(image.convert('L').resize((32, 32), Image.Resampling.LANCZOS), dtype=float)
    low = dctn(pixels, norm='ortho')[:8, :8].ravel()[1:]
    bits = np.packbits(low > np.median(low))
    return digest, bits

# Screen against every image under the original RoCoLe Photos folder.
rocole_photos = Path(split.image_path.iloc[0]).parent
references = sorted(p for p in rocole_photos.rglob('*') if p.is_file() and p.suffix.lower() in extensions)
assert {Path(p) for p in split.image_path}.issubset(set(references))
ref_hashes, ref_bits = zip(*(fingerprint(p) for p in references))
ref_bits = np.stack(ref_bits)
ref_exact = set(ref_hashes)
popcount = np.array([i.bit_count() for i in range(256)], dtype=np.uint8)
threshold = 6  # Conservative screen for candidate matches, fixed before scoring.
out = project / 'results/jmuben_external_evaluation' / uuid4().hex[:12]
out.mkdir(parents=True, exist_ok=False)
print('External evaluation folder:', out, flush=True)
audit = []
for index, (path, label) in enumerate(sources, 1):
    try:
        digest, bits = fingerprint(path)
    except (OSError, ValueError):
        audit.append(dict(image_path=str(path), label=label, status='unreadable'))
        continue
    distances = popcount[np.bitwise_xor(ref_bits, bits)].sum(axis=1)
    nearest = int(distances.argmin())
    status = 'exact_match' if digest in ref_exact else (
        'perceptual_candidate' if distances[nearest] <= threshold else 'included')
    audit.append(dict(image_path=str(path), label=label, status=status,
                      distance=int(distances[nearest]), nearest_rocole=str(references[nearest])))
    if index % 2000 == 0:
        print(f'Screened {index}/{len(sources)} JMuBEN images', flush=True)

audit = pd.DataFrame(audit)
audit.to_csv(out / 'overlap_audit.csv', index=False)
print(audit.groupby(['label', 'status']).size().to_string())
eligible = audit.loc[audit.status.eq('included')].reset_index(drop=True)
assert set(eligible.label) == set(class_folders)


In [ ]:
root = project / 'results/controlled_leakage_injection'
campaign = root / 'YOUR_REPEAT_CAMPAIGN'
matched = project / 'results/matched_augmentation_control/YOUR_MATCHED_CONTROL_RUN'
runs = {42: root / 'YOUR_SEED_42_RUN'}
for seed in (137, 911):
    candidates = list(campaign.glob(f'seed_{seed}_*'))
    assert len(candidates) == 1, candidates
    runs[seed] = candidates[0]

checkpoints = {}
for seed, run in runs.items():
    checkpoints[seed] = {
        'clean': run / 'dose_00/best.keras',
        'training_derived': matched / f'seed_{seed}/best.keras',
        'test_derived': run / 'dose_40/best.keras',
    }
assert all(p.is_file() for group in checkpoints.values() for p in group.values())
pd.DataFrame([(seed, name, str(path)) for seed, group in checkpoints.items()
              for name, path in group.items()], columns=['seed', 'condition', 'checkpoint']).to_csv(
    out / 'checkpoint_manifest.csv', index=False)

def decode(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image = tf.image.resize(image, (224, 224))
    return tf.keras.applications.mobilenet_v2.preprocess_input(image)

ds = tf.data.Dataset.from_tensor_slices(eligible.image_path.tolist())
from tempfile import TemporaryDirectory

# Cache the unchanged, preprocessed batches on local disk for all nine models.
image_cache = TemporaryDirectory(prefix="jmuben_cache_", dir="/content")
ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(32)
ds = ds.cache(str(Path(image_cache.name) / "images")).prefetch(tf.data.AUTOTUNE)
truth = eligible.label.eq('rust').to_numpy(dtype=int)
edges = np.linspace(0, 1, 11)
rows = []
predictions = eligible[['image_path', 'label']].copy()
for seed, group in checkpoints.items():
    for name, path in group.items():
        print(f'Evaluating {seed} {name}', flush=True)
        tf.keras.backend.clear_session()
        model = tf.keras.models.load_model(path, compile=False)
        prob = model.predict(ds, verbose=1).ravel()
        assert len(prob) == len(truth) and np.isfinite(prob).all()
        pred = (prob >= 0.5).astype(int)
        correct = pred == truth
        confidence = np.maximum(prob, 1 - prob)
        ece = 0.0
        for low, high in zip(edges[:-1], edges[1:]):
            mask = (confidence > low) & (confidence <= high)
            if mask.any():
                ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
        rows.append(dict(seed=seed, condition=name, n=len(truth), accuracy=correct.mean(),
                         macro_f1=f1_score(truth, pred, average='macro'), ece=ece))
        predictions[f'{seed}_{name}_rust_probability'] = prob
        pd.DataFrame(rows).to_csv(out / 'external_metrics.csv', index=False)
        predictions.to_csv(out / 'external_predictions.csv', index=False)

metrics = pd.DataFrame(rows)
paired = []
for seed, frame in metrics.groupby('seed'):
    frame = frame.set_index('condition')
    for first, second in [('test_derived','training_derived'), ('training_derived','clean'), ('test_derived','clean')]:
        for metric in ('accuracy', 'macro_f1', 'ece'):
            paired.append(dict(seed=seed, comparison=f'{first}_minus_{second}', metric=metric,
                               delta_pp=100 * (frame.loc[first, metric] - frame.loc[second, metric])))
paired = pd.DataFrame(paired)
paired.to_csv(out / 'paired_changes_by_seed.csv', index=False)
summary = paired.groupby(['comparison','metric']).delta_pp.agg(['mean','std','count'])
summary.to_csv(out / 'paired_summary.csv')
display(summary.round(3))
image_cache.cleanup()
print('Completed JMuBEN external evaluation:', out)
